In [ ]:
# Parámetros del job (fab job run ... -P corpus_version:string=0.2.0).
corpus_version = "0.2.0"

In [ ]:
# Cargador del plano de evaluación en Delta (WRK-TASK-102, DOC-RAG-003).
#
# Lee los ficheros aterrizados por fabric/load_evaluation.ps1 en
# Files/landing/<corpus_version>/ del Lakehouse de evaluación y los añade a tablas Delta:
#
# - load_log: una fila por fichero cargado (clave: sha256). Garantiza append-only idempotente: un
#   fichero ya cargado nunca se reescribe ni se vuelve a añadir.
# - corpus_manifest, gold_cases: manifiesto del corpus sintético y casos de gold set.
# - evaluation_runs, evaluation_profiles, evaluation_cases: informes de benchmark (schema 1.0/1.1).
#
# Salvaguardas: cada gold set e informe debe declarar el corpus_version del manifiesto de
# compatibilidad; cada informe debe referenciar el sha256 del manifiesto aterrizado (corpus
# sintético); se rechaza cualquier informe con campos de contenido documental (answer, snippet,
# text, claims). Nunca se usa overwrite ni merge.

import hashlib
import json
from datetime import UTC, datetime

import notebookutils
import yaml
from pyspark.sql import Row, SparkSession
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

FORBIDDEN_KEYS = {"answer", "snippet", "text", "claims"}
LEGACY_BACKEND = ("qdrant", "hnsw")

fs = notebookutils.fs
try:
    lakehouse = notebookutils.variableLibrary.getLibrary("ragdocs_params").lakehouse_name
except Exception:
    lakehouse = "ragdocs_eval"
workspace = notebookutils.runtime.context["currentWorkspaceName"]
base = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse"
landing = f"{base}/Files/landing/{corpus_version}"
loaded_at = datetime.now(UTC).replace(tzinfo=None)
spark = SparkSession.builder.getOrCreate()


def S(*fields):
    return StructType([StructField(name, kind, True) for name, kind in fields])


SCHEMAS = {
    "load_log": S(
        ("source_sha256", StringType()),
        ("kind", StringType()),
        ("corpus_version", StringType()),
        ("source_file", StringType()),
        ("loaded_at", TimestampType()),
    ),
    "corpus_manifest": S(
        ("corpus_version", StringType()),
        ("relative_path", StringType()),
        ("sha256", StringType()),
        ("manifest_sha256", StringType()),
        ("loaded_at", TimestampType()),
    ),
    "gold_cases": S(
        ("corpus_version", StringType()),
        ("gold_set", StringType()),
        ("split", StringType()),
        ("case_id", StringType()),
        ("category", StringType()),
        ("question", StringType()),
        ("expected_status", StringType()),
        ("expected_language", StringType()),
        ("expected_documents", ArrayType(StringType())),
        ("required_facts_json", StringType()),
        ("source_sha256", StringType()),
        ("loaded_at", TimestampType()),
    ),
    "evaluation_runs": S(
        ("run_id", StringType()),
        ("corpus_version", StringType()),
        ("benchmark_id", StringType()),
        ("phase", StringType()),
        ("report_schema_version", StringType()),
        ("source_file", StringType()),
        ("started_at", StringType()),
        ("completed_at", StringType()),
        ("revision", StringType()),
        ("config_sha256", StringType()),
        ("corpus_manifest_sha256", StringType()),
        ("loaded_at", TimestampType()),
    ),
    "evaluation_profiles": S(
        ("run_id", StringType()),
        ("corpus_version", StringType()),
        ("profile_id", StringType()),
        ("vector_backend", StringType()),
        ("vector_search_mode", StringType()),
        ("retrieval_strategy", StringType()),
        ("reranker_model", StringType()),
        ("index_fingerprint_digest", StringType()),
        ("embedding_model", StringType()),
        ("embedding_revision", StringType()),
        ("generator_mode", StringType()),
        ("passed", IntegerType()),
        ("total", IntegerType()),
        ("score", DoubleType()),
        ("recall_at_1", DoubleType()),
        ("recall_at_3", DoubleType()),
        ("recall_at_5", DoubleType()),
        ("recall_at_8", DoubleType()),
        ("reciprocal_rank", DoubleType()),
        ("precision_at_1", DoubleType()),
        ("precision_at_8", DoubleType()),
        ("retrieval_p50_ms", DoubleType()),
        ("retrieval_p95_ms", DoubleType()),
        ("total_p50_ms", DoubleType()),
        ("total_p95_ms", DoubleType()),
        ("indexing_ms", DoubleType()),
        ("loaded_at", TimestampType()),
    ),
    "evaluation_cases": S(
        ("run_id", StringType()),
        ("corpus_version", StringType()),
        ("profile_id", StringType()),
        ("vector_backend", StringType()),
        ("case_id", StringType()),
        ("passed", BooleanType()),
        ("status_ok", BooleanType()),
        ("retrieval_ok", BooleanType()),
        ("facts_ok", BooleanType()),
        ("citations_ok", BooleanType()),
        ("failure_stage", StringType()),
        ("recall_at_8", DoubleType()),
        ("reciprocal_rank", DoubleType()),
        ("retrieval_ms", DoubleType()),
        ("total_ms", DoubleType()),
        ("loaded_at", TimestampType()),
    ),
}


def table_path(name):
    return f"{base}/Tables/{name}"


def read_text(path):
    return fs.head(path, 64 * 1024 * 1024)


def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def loaded_hashes():
    try:
        return {row.source_sha256 for row in spark.read.format("delta").load(
            table_path("load_log")).select("source_sha256").collect()}
    except Exception:
        return set()


def append(name, rows):
    if rows:
        frame = spark.createDataFrame([Row(**row) for row in rows], SCHEMAS[name])
        frame.write.format("delta").mode("append").save(table_path(name))


def as_float(value):
    return None if value is None else float(value)


def forbidden_keys(value, path="$"):
    if isinstance(value, dict):
        for key, item in value.items():
            if key in FORBIDDEN_KEYS:
                yield f"{path}.{key}"
            yield from forbidden_keys(item, f"{path}.{key}")
    elif isinstance(value, list):
        for index, item in enumerate(value):
            yield from forbidden_keys(item, f"{path}[{index}]")


def backend_of(report, profile):
    for source in (profile, report):
        if source.get("vector_backend") and source.get("vector_search_mode"):
            return source["vector_backend"], source["vector_search_mode"]
    if report.get("schema_version", "1.0") == "1.0":
        return LEGACY_BACKEND
    raise ValueError("Informe 1.1 sin vector_backend/vector_search_mode")


already = loaded_hashes()
summary = {"loaded": [], "skipped": []}

compatibility_text = read_text(f"{landing}/corpus-compatibility.yaml")
compatibility = yaml.safe_load(compatibility_text)
if compatibility["corpus_version"] != corpus_version:
    raise ValueError("El manifiesto de compatibilidad no corresponde al corpus_version del job")

manifest_text = read_text(f"{landing}/manifest.sha256")
manifest_sha = sha256(manifest_text)
log_rows = []

if manifest_sha not in already:
    append("corpus_manifest", [
        {"corpus_version": corpus_version, "relative_path": relative, "sha256": digest,
         "manifest_sha256": manifest_sha, "loaded_at": loaded_at}
        for digest, relative in (
            line.split("  ", 1) for line in manifest_text.splitlines()
            if line and not line.startswith("#")
        )
    ])
    log_rows.append({"source_sha256": manifest_sha, "kind": "corpus_manifest",
                     "corpus_version": corpus_version, "source_file": "manifest.sha256",
                     "loaded_at": loaded_at})
    summary["loaded"].append("manifest.sha256")
else:
    summary["skipped"].append("manifest.sha256")

for entry in fs.ls(f"{landing}/gold-sets"):
    text = read_text(entry.path)
    digest = sha256(text)
    if digest in already:
        summary["skipped"].append(entry.name)
        continue
    gold = yaml.safe_load(text)
    if gold.get("corpus_version") != corpus_version:
        raise ValueError(f"{entry.name} declara otro corpus_version")
    append("gold_cases", [
        {"corpus_version": corpus_version, "gold_set": entry.name, "split": gold.get("split"),
         "case_id": case["id"], "category": case.get("category"), "question": case["question"],
         "expected_status": case["expected_status"],
         "expected_language": case.get("expected_language"),
         "expected_documents": list(case.get("expected_documents") or []),
         "required_facts_json": json.dumps(case.get("required_facts") or [], ensure_ascii=False),
         "source_sha256": digest, "loaded_at": loaded_at}
        for case in gold["cases"]
    ])
    log_rows.append({"source_sha256": digest, "kind": "gold_set", "corpus_version": corpus_version,
                     "source_file": entry.name, "loaded_at": loaded_at})
    summary["loaded"].append(entry.name)

for entry in fs.ls(f"{landing}/reports"):
    text = read_text(entry.path)
    digest = sha256(text)
    if digest in already:
        summary["skipped"].append(entry.name)
        continue
    report = json.loads(text)
    leaks = list(forbidden_keys(report))
    if leaks:
        raise ValueError(f"{entry.name} contiene campos de contenido documental: {leaks[:5]}")
    if report.get("corpus_version") != corpus_version:
        raise ValueError(f"{entry.name} declara otro corpus_version")
    if report.get("corpus_manifest_sha256") != manifest_sha:
        raise ValueError(f"{entry.name} no referencia el manifiesto del corpus sintético")
    run_id = digest[:16]
    append("evaluation_runs", [{
        "run_id": run_id, "corpus_version": corpus_version,
        "benchmark_id": report.get("benchmark_id"), "phase": report.get("phase"),
        "report_schema_version": report.get("schema_version", "1.0"), "source_file": entry.name,
        "started_at": report.get("started_at"), "completed_at": report.get("completed_at"),
        "revision": report.get("revision"), "config_sha256": report.get("config_sha256"),
        "corpus_manifest_sha256": report.get("corpus_manifest_sha256"), "loaded_at": loaded_at,
    }])
    profiles, cases = [], []
    for profile in report["profiles"]:
        backend, mode = backend_of(report, profile)
        retrieval = profile.get("retrieval") or {}
        performance = profile.get("performance") or {}
        config = profile.get("effective_config") or {}
        profiles.append({
            "run_id": run_id, "corpus_version": corpus_version,
            "profile_id": profile["profile_id"], "vector_backend": backend,
            "vector_search_mode": mode, "retrieval_strategy": profile.get("retrieval_strategy"),
            "reranker_model": profile.get("reranker_model"),
            "index_fingerprint_digest": (profile.get("index_fingerprint") or {}).get("digest"),
            "embedding_model": config.get("embedding_model"),
            "embedding_revision": config.get("embedding_revision"),
            "generator_mode": config.get("generator_mode"),
            "passed": profile.get("passed"), "total": profile.get("total"),
            "score": as_float(profile.get("score")),
            **{name: as_float(retrieval.get(name)) for name in (
                "recall_at_1", "recall_at_3", "recall_at_5", "recall_at_8", "reciprocal_rank",
                "precision_at_1", "precision_at_8")},
            "retrieval_p50_ms": as_float((performance.get("retrieval") or {}).get("p50_ms")),
            "retrieval_p95_ms": as_float((performance.get("retrieval") or {}).get("p95_ms")),
            "total_p50_ms": as_float((performance.get("total") or {}).get("p50_ms")),
            "total_p95_ms": as_float((performance.get("total") or {}).get("p95_ms")),
            "indexing_ms": as_float(profile.get("indexing_ms")), "loaded_at": loaded_at,
        })
        for case in profile.get("cases") or []:
            metrics = case.get("retrieval_metrics") or {}
            stages = case.get("stages") or {}
            cases.append({
                "run_id": run_id, "corpus_version": corpus_version,
                "profile_id": profile["profile_id"], "vector_backend": backend,
                "case_id": case["id"], "passed": case.get("passed"),
                "status_ok": case.get("status_ok"), "retrieval_ok": case.get("retrieval_ok"),
                "facts_ok": case.get("facts_ok"), "citations_ok": case.get("citations_ok"),
                "failure_stage": case.get("failure_stage"),
                "recall_at_8": as_float(metrics.get("recall_at_8")),
                "reciprocal_rank": as_float(metrics.get("reciprocal_rank")),
                "retrieval_ms": as_float(stages.get("retrieval_ms")),
                "total_ms": as_float(stages.get("total_ms")), "loaded_at": loaded_at,
            })
    append("evaluation_profiles", profiles)
    append("evaluation_cases", cases)
    log_rows.append({"source_sha256": digest, "kind": "benchmark_report",
                     "corpus_version": corpus_version, "source_file": entry.name,
                     "loaded_at": loaded_at})
    summary["loaded"].append(entry.name)

append("load_log", log_rows)
counts = {name: spark.read.format("delta").load(table_path(name)).count()
          for name in SCHEMAS if fs.exists(table_path(name))}
result = json.dumps({"corpus_version": corpus_version, **summary, "row_counts": counts})
print(result)
notebookutils.notebook.exit(result)